In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500
emergency_types = ['Accident', 'Cardiac', 'Fire', 'Fall', 'Breathing', 'Assault', 'Other']
severities = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

data = pd.DataFrame({
    'emergency_type': np.random.choice(emergency_types, n),
    'severity': np.random.choice(severities, n, p=[0.3, 0.35, 0.25, 0.10]),
    'patient_age': np.random.randint(1, 90, n),
    'hour_of_day': np.random.randint(0, 24, n),
    'distance_to_nearest_hospital_km': np.round(np.random.uniform(0.5, 20, n), 2),
})

# Priority score: higher for CRITICAL/HIGH severity, very young/old patients, and late-night hours
severity_weight = data['severity'].map({'LOW': 0.2, 'MEDIUM': 0.5, 'HIGH': 0.75, 'CRITICAL': 0.95})
age_weight = np.where((data['patient_age'] < 5) | (data['patient_age'] > 65), 0.15, 0)
night_weight = np.where((data['hour_of_day'] < 6) | (data['hour_of_day'] > 22), 0.1, 0)

data['priority_score'] = np.clip(severity_weight + age_weight + night_weight + np.random.normal(0, 0.03, n), 0, 1)

data.head(10)

,emergency_type,severity,patient_age,hour_of_day,distance_to_nearest_hospital_km,priority_score
0,Other,MEDIUM,82,5,9.17,0.736070
1,Fall,MEDIUM,89,6,0.79,0.639617
2,Breathing,MEDIUM,60,4,11.95,0.621854
3,Other,HIGH,43,10,3.78,0.744769
4,Fire,LOW,76,12,13.03,0.340430
5,Breathing,CRITICAL,68,18,15.31,1.000000
6,Breathing,MEDIUM,5,22,10.26,0.526590
7,Other,HIGH,37,9,11.08,0.729220
8,Cardiac,LOW,72,9,19.05,0.348004
9,Fire,MEDIUM,31,10,17.02,0.507358


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Features and target
X = data[['emergency_type', 'severity', 'patient_age', 'hour_of_day', 'distance_to_nearest_hospital_km']]
y = data['priority_score']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One-hot encode the text columns, leave numeric columns as-is
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['emergency_type', 'severity']),
], remainder='passthrough')

# Build a pipeline: preprocess -> train a Random Forest model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42)),
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error: {mae:.4f}")
print(f"(This means predictions are off by about {mae:.2%} on average)")

Mean Absolute Error: 0.0319
(This means predictions are off by about 3.19% on average)


In [3]:
import joblib

joblib.dump(model, '../trained_models/emergency_priority_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [4]:
# Test: a CRITICAL cardiac emergency, elderly patient, at 3am, far from hospital
test_case = pd.DataFrame({
    'emergency_type': ['Cardiac'],
    'severity': ['CRITICAL'],
    'patient_age': [78],
    'hour_of_day': [3],
    'distance_to_nearest_hospital_km': [15.0],
})

predicted_priority = model.predict(test_case)[0]
print(f"Predicted priority score: {predicted_priority:.3f}")

# Compare: a LOW severity case, young adult, midday, close to hospital
test_case_2 = pd.DataFrame({
    'emergency_type': ['Fall'],
    'severity': ['LOW'],
    'patient_age': [25],
    'hour_of_day': [14],
    'distance_to_nearest_hospital_km': [2.0],
})

predicted_priority_2 = model.predict(test_case_2)[0]
print(f"Predicted priority score: {predicted_priority_2:.3f}")

Predicted priority score: 1.000
Predicted priority score: 0.203
